# Color Palette Analyzer — Workflow en Python

Analiza los colores principales de una imagen en **Google Drive**, usando **Azure AI Foundry**.
Genera un `.zip` con:
- `palette_chart.png` — paleta visual de los 5 colores
- `analisis.txt` — análisis del director de arte
- `analisis_audio.mp3` — audio del análisis (ElevenLabs)


## 1) Instalación y dependencias

Librerías necesarias que no están disponibles por defecto en Colab.

In [19]:
!pip install agent-framework-core
!pip install agent-framework-foundry
!pip install elevenlabs requests pillow

## 2) Credenciales

Rellena las variables antes de ejecutar. No subas este archivo con las keys.

In [20]:
FOUNDRY_ENDPOINT  = "https://n8nprueba-resource.services.ai.azure.com/"              # https://<proyecto>.services.ai.azure.com/
FOUNDRY_API_KEY   = "Fhyf30hJicRBBXThBvTBLNfdNtxco39E3ld4ByG9h8VYM1RJCoMBJQQJ99CFACfhMk5XJ3w3AAAAACOGJMTZ"              # Portal Azure AI Foundry > Keys and Endpoint
FOUNDRY_MODEL     = "gpt-4o-mini"

ELEVENLABS_API_KEY  = "7469f8785d47cbd4daeb9b8d722316f9542c79aef026cb016c132c9abb8cf7c2"                       # elevenlabs.io > Profile > API Key
ELEVENLABS_VOICE_ID = "IKne3meq5aSn9XLyUdCD"  # Charlie

assert FOUNDRY_ENDPOINT, "Falta FOUNDRY_ENDPOINT"
assert FOUNDRY_API_KEY,  "Falta FOUNDRY_API_KEY"

print("Credenciales cargadas")

Credenciales cargadas


## 3) Google Drive

Monta el Drive para acceder a la imagen de entrada. Colab pedirá autorización al ejecutar.

In [21]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive montado en /content/drive


## 4) Cliente y agente de Azure AI Foundry

Configura el agente con su system prompt. `ApiKeyCredential` es un adaptador necesario
porque `FoundryChatClient` espera la interfaz OAuth de Azure, no una API key directa.

In [22]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.core.credentials import AccessToken
import time

# FoundryChatClient requiere un objeto con get_token(); este wrapper adapta una API key a esa interfaz
class ApiKeyCredential:
    def __init__(self, api_key: str):
        self._api_key = api_key

    def get_token(self, *scopes, **kwargs) -> AccessToken:
        return AccessToken(self._api_key, int(time.time()) + 3600)


SYSTEM_PROMPT = (
    "Actúa como un director de arte experto en colorimetría. "
    "Analiza la imagen y extrae los 5 colores principales. "
    "Después, evalúa la paleta actual y sugiere 2 paletas alternativas "
    "explicando por qué mejorarían el diseño.\n"
    "Devuelve ÚNICAMENTE un objeto JSON con esta estructura exacta, sin texto fuera del JSON:\n"
    "{\n"
    '"colores": [\n'
    '{"hex": "E3C5A8", "codigo": "14-1217 TCX", "nombre": "Amberlight"},\n'
    '{"hex": "B5C7D6", "codigo": "14-4112 TCX", "nombre": "Skyway"}\n'
    "],\n"
    '"analisis": "Tu opinión de la paleta original y tus 2 nuevas propuestas."\n'
    "}\n"
    "Regla: Extrae entre 3 y 6 colores según los que realmente aparezcan en la imagen. Los hex no deben llevar la almohadilla (#)."
)

_foundry_client = FoundryChatClient(
    project_endpoint=FOUNDRY_ENDPOINT,
    model=FOUNDRY_MODEL,
    credential=ApiKeyCredential(FOUNDRY_API_KEY),
)

color_agent = Agent(
    client=_foundry_client,
    name="ColorPaletteAgent",
    instructions=SYSTEM_PROMPT,
)

print("FoundryChatClient y Agent listos")

FoundryChatClient y Agent listos


## 5) Análisis de imagen

Codifica la imagen en base64 y la envía al modelo en formato multimodal.
Se usa `AzureOpenAI` directamente porque el SDK de MAF no soporta `image_url` nativo.

In [23]:
import base64
import json
import re
from pathlib import Path
from openai import AzureOpenAI


def load_image_as_base64(image_path: str) -> tuple[str, str]:
    """Carga una imagen local y devuelve (base64_data, media_type)."""
    ext = Path(image_path).suffix.lower()
    media_type_map = {
        ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
        ".png": "image/png",  ".gif": "image/gif", ".webp": "image/webp",
    }
    media_type = media_type_map.get(ext, "image/jpeg")
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8"), media_type


async def analyze_image(image_path: str) -> dict:
    """
    Envía la imagen al modelo usando la API de visión directamente
    con el formato multimodal correcto (content array con image_url).
    """
    b64_data, media_type = load_image_as_base64(image_path)

    client = AzureOpenAI(
        api_key=FOUNDRY_API_KEY,
        azure_endpoint=FOUNDRY_ENDPOINT,
        api_version="2024-02-15-preview",
    )

    response = client.chat.completions.create(
        model=FOUNDRY_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            # Data URL: incrusta la imagen en la petición sin servidor intermedio
                            "url": f"data:{media_type};base64,{b64_data}"
                        },
                    },
                    {
                        "type": "text",
                        "text": "Analiza esta imagen y devuelve el JSON."
                    },
                ],
            },
        ],
        max_tokens=1000,
    )

    ai_text = response.choices[0].message.content
    # El modelo puede envolver la respuesta en bloques markdown aunque se le pida que no
    ai_text_clean = re.sub(r"```json|```", "", ai_text).strip()
    return json.loads(ai_text_clean)


print("Funcion analyze_image lista")

Funcion analyze_image lista


## 6) Generación de la paleta visual

Construye la URL de QuickChart con los colores extraídos por el agente.
El formatter de datalabels debe ser código JS nativo, por eso se inyecta como string después de serializar.

In [24]:
import urllib.parse


def build_chart_url(colores: list[dict]) -> str:
    """
    Equivalente al nodo 'Code in JavaScript' de n8n.
    Genera la URL de QuickChart con la paleta de 5 colores.
    """
    hex_codes = ["#" + c["hex"].lstrip("#") for c in colores]
    labels    = [["ABB", c["codigo"]] for c in colores]  # 2 líneas por barra, igual que el JS

    chart_config = {
        "type": "bar",
        "data": {
            "labels": labels,
            "datasets": [{
                "data": [1, 1, 1, 1, 1],
                "backgroundColor": hex_codes,
                "barPercentage": 1.0,
                "categoryPercentage": 1.0,
            }],
        },
        "options": {
            "layout": {"padding": 0},
            "legend": {"display": False},
            "scales": {
                "xAxes": [{"display": False}],
                "yAxes": [{"display": False, "ticks": {"min": 0, "max": 1}}],
            },
            "plugins": {
                "datalabels": {
                    "color": "#000000",
                    "backgroundColor": "#ffffff",
                    "borderRadius": 4,
                    "padding": 12,
                    "rotation": 90,
                    "anchor": "end",
                    "align": "start",
                    "offset": 40,
                    "font": {"family": "sans-serif", "size": 16, "weight": "bold"},
                    # Placeholder sustituido por la función JS real justo después
                    "formatter": "INYECCION_DE_CODIGO",
                },
            },
        },
    }

    # Inyección de función JS — réplica exacta del replace del nodo Code de n8n
    config_str = json.dumps(chart_config)
    config_str = config_str.replace(
        '"INYECCION_DE_CODIGO"',
        "function(value, context) { return context.chart.data.labels[context.dataIndex]; }",
    )

    return "https://quickchart.io/chart?w=800&h=600&bkg=white&c=" + urllib.parse.quote(config_str)


print("Funcion build_chart_url lista")

Funcion build_chart_url lista


## 7) Descarga de la imagen de paleta

Hace una petición GET a QuickChart y escribe la imagen PNG resultante en disco.

In [25]:
import requests


def download_chart_image(chart_url: str, output_path: str = "palette_chart.png") -> str:
    """
    Equivalente al nodo 'HTTP Request' de n8n (responseFormat: file).
    Descarga la imagen de QuickChart y la guarda en disco.
    """
    response = requests.get(chart_url, timeout=30)
    response.raise_for_status()
    Path(output_path).write_bytes(response.content)
    print(f"Paleta guardada en: {output_path}")
    return output_path


print("Funcion download_chart_image lista")

Funcion download_chart_image lista


## 8) Text-to-Speech con ElevenLabs

Convierte el análisis a audio MP3. La API devuelve un generador de chunks
que se concatenan en memoria antes de escribir el archivo.

In [26]:
from elevenlabs.client import ElevenLabs


def elevenlabs_tts(text: str, output_path: str = "analisis_audio.mp3") -> str:
    """
    Equivalente al nodo 'Convert text to speech' de n8n.
    Convierte el texto de análisis a audio con la voz Charlie de ElevenLabs.
    """
    el_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

    audio_generator = el_client.text_to_speech.convert(
        voice_id=ELEVENLABS_VOICE_ID,
        text=text,
        model_id="eleven_multilingual_v2",
    )

    audio_bytes = b"".join(
        chunk if isinstance(chunk, bytes) else bytes(chunk)
        for chunk in audio_generator
    )

    Path(output_path).write_bytes(audio_bytes)
    print(f"Audio guardado en: {output_path}")
    return output_path


print("Funcion elevenlabs_tts lista")

Funcion elevenlabs_tts lista


## 9) Empaquetado en ZIP

Reúne los tres archivos en un ZIP y lo descarga al navegador con `files.download()`.

In [27]:
import shutil
import tempfile
import zipfile
from pathlib import Path


def build_zip(
    chart_path: str,
    analisis: str,
    audio_path: str | None,
    zip_path: str = "/content/resultado_paleta.zip",
) -> str:
    """
    Empaqueta los 3 entregables en un ZIP y lo descarga desde Colab.
      - palette_chart.png
      - analisis.txt
      - analisis_audio.mp3  (solo si se genero audio)
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp = Path(tmpdir)

        shutil.copy(chart_path, tmp / "palette_chart.png")
        (tmp / "analisis.txt").write_text(analisis, encoding="utf-8")
        if audio_path and Path(audio_path).exists():
            shutil.copy(audio_path, tmp / "analisis_audio.mp3")

        zip_base = zip_path.removesuffix(".zip")
        shutil.make_archive(zip_base, "zip", root_dir=tmpdir)

    final = zip_base + ".zip"

    print("Contenido del ZIP:")
    with zipfile.ZipFile(final) as zf:
        for name in zf.namelist():
            size = zf.getinfo(name).file_size
            print(f"   {name:30s}  {size / 1024:.1f} KB")

    from google.colab import files
    files.download(final)
    print(f"ZIP descargado: {final}")
    return final


print("Funcion build_zip lista")

Funcion build_zip lista


## 10) Workflow completo

Orquesta todos los pasos en secuencia. El decorador `@workflow` de MAF
requiere un único parámetro de entrada, por eso se usa un dict.

In [28]:
from agent_framework import workflow


@workflow
async def color_palette_workflow(request: dict) -> dict:
    """
    Workflow completo.
    Claves del dict de entrada:
      - image_path     (str)  ruta a la imagen en Drive
      - generate_audio (bool) activar ElevenLabs, default True
      - zip_output     (str)  ruta del ZIP de salida
    """
    image_path     = request["image_path"]
    generate_audio = request.get("generate_audio", True)
    zip_output     = request.get("zip_output", "/content/resultado_paleta.zip")

    print(f"Iniciando workflow | imagen: {image_path}")

    ai_data  = await analyze_image(image_path)
    colores  = ai_data["colores"]
    analisis = ai_data["analisis"]
    print(f"Colores extraidos: {[c['nombre'] for c in colores]}")

    chart_url  = build_chart_url(colores)
    chart_path = download_chart_image(chart_url, output_path="/content/palette_chart.png")

    audio_path = None
    if generate_audio:
        audio_path = elevenlabs_tts(analisis, output_path="/content/analisis_audio.mp3")

    build_zip(chart_path, analisis, audio_path, zip_path=zip_output)

    print("Workflow completado")
    return {"colores": colores, "analisis": analisis,
            "chart_path": chart_path, "audio_path": audio_path}


print("Workflow definido")

Workflow definido


/tmp/ipykernel_7467/2036767569.py:4: ExperimentalWarning: [FUNCTIONAL_WORKFLOWS] workflow is experimental and may change or be removed in future versions without notice.


## 11) Ejecución

Cambia `IMAGE_PATH` con la ruta de tu imagen en Drive y ejecuta la celda.

In [29]:
IMAGE_PATH = "/content/drive/MyDrive/fotoanalisis.png"

result = await color_palette_workflow.run({
    "image_path":     IMAGE_PATH,
    "generate_audio": True,           # ← False si no tienes clave de ElevenLabs
    "zip_output":     "/content/resultado_paleta.zip",
})

resultado = result.get_outputs()[0]

Iniciando workflow | imagen: /content/drive/MyDrive/fotoanalisis.png
Colores extraidos: ['Tropical Sea', 'Horizon Blue', 'Almond Oil', 'Lime Green']
Paleta guardada en: /content/palette_chart.png
Audio guardado en: /content/analisis_audio.mp3
Contenido del ZIP:
   palette_chart.png               28.9 KB
   analisis.txt                    0.4 KB
   analisis_audio.mp3              418.4 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ZIP descargado: /content/resultado_paleta.zip
Workflow completado
